Load logic for `adwm_wh.gold.DimCustomer`.

Business grain:

* One row per `CustomerID`
* `CustomerID` is the business key used by the merge
* Person and related source records are deduplicated before the merge input is built

Source tables:

* `adwm_wh.silver.customer`
* `adwm_wh.silver.person`
* `adwm_wh.silver.store`
* `adwm_wh.silver.salesterritory`

Notebook contents:

* Upsert logic for the gold dimension
* Validation summary comparing total rows to distinct `CustomerID`
* Validation detail query that lists duplicate `CustomerID` values only if any exist

Operational note:

* `modified_date` is refreshed only when tracked customer attributes change

In [0]:
%sql
WITH person_dedup AS (
    SELECT
        BusinessEntityID,
        FirstName,
        LastName
    FROM (
        SELECT
            BusinessEntityID,
            FirstName,
            LastName,
            ROW_NUMBER() OVER (
                PARTITION BY BusinessEntityID
                ORDER BY CASE
                    WHEN upper(coalesce(FirstName, '')) = 'UNKNOWN'
                     AND upper(coalesce(LastName, '')) = 'UNKNOWN' THEN 1
                    ELSE 0
                END,
                FirstName,
                LastName
            ) AS rn
        FROM adwm_wh.silver.person
    ) p
    WHERE rn = 1
),
customer_source AS (
    SELECT
        c.CustomerID,
        CASE WHEN c.PersonID IS NOT NULL THEN 'Person' ELSE 'Store' END AS CustomerType,
        COALESCE(NULLIF(trim(concat_ws(' ', p.FirstName, p.LastName)), ''), s.Name) AS FullName,
        c.AccountNumber,
        st.Name AS TerritoryName,
        st.CountryRegionCode AS CountryRegion,
        sha2(
            concat_ws(
                '||',
                coalesce(CASE WHEN c.PersonID IS NOT NULL THEN 'Person' ELSE 'Store' END, ''),
                coalesce(COALESCE(NULLIF(trim(concat_ws(' ', p.FirstName, p.LastName)), ''), s.Name), ''),
                coalesce(c.AccountNumber, ''),
                coalesce(st.Name, ''),
                coalesce(st.CountryRegionCode, '')
            ),
            256
        ) AS row_hash
    FROM adwm_wh.silver.customer c
    LEFT JOIN person_dedup p
        ON p.BusinessEntityID = c.PersonID
    LEFT JOIN adwm_wh.silver.store s
        ON s.BusinessEntityID = c.StoreID
    LEFT JOIN adwm_wh.silver.salesterritory st
        ON st.TerritoryID = c.TerritoryID
)
MERGE INTO adwm_wh.gold.DimCustomer AS target
USING customer_source AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED AND sha2(
    concat_ws(
        '||',
        coalesce(target.CustomerType, ''),
        coalesce(target.FullName, ''),
        coalesce(target.AccountNumber, ''),
        coalesce(target.TerritoryName, ''),
        coalesce(target.CountryRegion, '')
    ),
    256
) <> source.row_hash THEN UPDATE SET
    target.CustomerType = source.CustomerType,
    target.FullName = source.FullName,
    target.AccountNumber = source.AccountNumber,
    target.TerritoryName = source.TerritoryName,
    target.CountryRegion = source.CountryRegion,
    target.modified_date = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    CustomerID,
    CustomerType,
    FullName,
    AccountNumber,
    TerritoryName,
    CountryRegion,
    modified_date
)
VALUES (
    source.CustomerID,
    source.CustomerType,
    source.FullName,
    source.AccountNumber,
    source.TerritoryName,
    source.CountryRegion,
    current_timestamp()
);

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CustomerID) AS distinct_customer_ids,
    COUNT(*) - COUNT(DISTINCT CustomerID) AS duplicate_row_count
FROM adwm_wh.gold.DimCustomer;

In [0]:
%sql
SELECT
    CustomerID,
    COUNT(*) AS row_count,
    MIN(modified_date) AS first_modified_date,
    MAX(modified_date) AS last_modified_date
FROM adwm_wh.gold.DimCustomer
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY row_count DESC, CustomerID;